<a href="https://colab.research.google.com/github/kikolorenzo/kikolorenzo/blob/main/ChatNtry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn pandas InstructorEmbedding sentence-transformers transformers scikit-learn torch pyngrok
!pip install git+https://github.com/KillianLucas/open-interpreter.git

  Cloning https://github.com/KillianLucas/open-interpreter.git to /tmp/pip-req-build-2xvck4we
  Running command git clone --filter=blob:none --quiet https://github.com/KillianLucas/open-interpreter.git /tmp/pip-req-build-2xvck4we
  Resolved https://github.com/KillianLucas/open-interpreter.git to commit dcfcffa9af13737bee1fa128364a718b56324998
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached starlette-0.37.2-py3-none-any.whl.metadata (5.9 kB)
Using cached starlette-0.37.2-py3-none-any.whl (71 kB)
  Created wheel for open-interpreter: filename=open_interpreter-0.4.3-py3-none-any.whl size=255517 sha256=3bef55573b4651cd7496c923c8dab93be7bd6aeec4bf918e2f28ee1accbabb53
  Stored in directory: /tmp/pip-ephem-wheel-cache-932usfih/wheels/12/b3/96/1fbfc8772afa909f9c442a0095abb3a7d51fa889aacbf46a91
Successfully built open-interpreter
  Attempting uninstall: starlette
    Found existing installation

In [2]:
!pip install "fastapi==0.103.2"  # compatible con starlette 0.37.2


  Using cached starlette-0.27.0-py3-none-any.whl.metadata (5.8 kB)
Using cached starlette-0.27.0-py3-none-any.whl (66 kB)
  Attempting uninstall: starlette
    Found existing installation: starlette 0.37.2
    Uninstalling starlette-0.37.2:
      Successfully uninstalled starlette-0.37.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
open-interpreter 0.4.3 requires starlette<0.38.0,>=0.37.2, but you have starlette 0.27.0 which is incompatible.


In [3]:
!pip uninstall open-interpreter -y

Found existing installation: open-interpreter 0.4.3
Uninstalling open-interpreter-0.4.3:
  Successfully uninstalled open-interpreter-0.4.3


In [4]:
!pip install python-multipart


In [5]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import pandas as pd
import io
import nest_asyncio
import uvicorn

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class EmbedRequest(BaseModel):
    text: str

class GenerateRequest(BaseModel):
    question: str
    csv_content: str

@app.post("/embed")
async def generate_embedding(req: EmbedRequest):
    from InstructorEmbedding import INSTRUCTOR
    model = INSTRUCTOR("hkunlp/instructor-base")
    instruction = "Represent the sentence for semantic search:"
    embedding = model.encode([[instruction, req.text]])[0].tolist()
    return {"embedding": embedding}

@app.post("/generate")
async def generate_answer(req: GenerateRequest):
    df = pd.read_csv(io.StringIO(req.csv_content))
    question = req.question

    import ast
    import contextlib
    import io as sysio

    namespace = {"df": df}
    stdout = sysio.StringIO()

    # Simulamos un agente simple que interpreta instrucciones
    if "producto más vendido" in question.lower():
        code = "df.groupby('Producto')['Unidades'].sum().idxmax()"
    elif "menos se vende" in question.lower():
        code = "df.groupby('Producto')['Unidades'].sum().idxmin()"
    elif "más ingresos" in question.lower():
        code = "df.groupby('Comercial')['Ventas'].sum().idxmax()"
    elif "más devoluciones" in question.lower():
        code = "df.groupby('Comercial')['Devoluciones'].sum().idxmax()"
    elif "provincia" in question.lower():
        code = "df.groupby('Provincia')['Ventas'].sum().idxmax()"
    else:
        return {"answer": "No se puede responder automáticamente a esa pregunta. Intenta reformularla."}

    with contextlib.redirect_stdout(stdout):
        result = eval(code, namespace)

    return {"answer": str(result)}


@app.post("/ingest-csv")
async def ingest_csv(file: UploadFile = File(...)):
    df = pd.read_csv(file.file)
    return {"rows": len(df), "status": "ok"}


In [6]:
import threading

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

nest_asyncio.apply()
thread = threading.Thread(target=run)
thread.start()


In [8]:
!ngrok config add-authtoken 2xrKBeK2iHY7BpnmRlg8aVVXK2L_2bbaHnFUGK2ZMgVKbgutL

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [9]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)
print("🔗 Tu backend está accesible en:")
print(public_url)


🔗 Tu backend está accesible en:
NgrokTunnel: "https://0a38-34-150-190-77.ngrok-free.app" -> "http://localhost:8000"
